# Explanation Quality and User Decision Lab


## Purpose

This lab models explanation quality as a composite of fidelity, usefulness, actionability, and stability.

Learning goals:

- Simulate explanation-quality dimensions.
- Compute an explanation-quality score.
- Explore how explanation quality affects user decisions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 1000

explanations = pd.DataFrame({
    "case_id": [f"C-{i:04d}" for i in range(1, n + 1)],
    "fidelity": rng.beta(5, 2, size=n),
    "usefulness": rng.beta(4, 3, size=n),
    "actionability": rng.beta(3.5, 3, size=n),
    "stability": rng.beta(4, 2.5, size=n),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
})

weights = {
    "fidelity": 0.35,
    "usefulness": 0.25,
    "actionability": 0.25,
    "stability": 0.15,
}

explanations["explanation_quality"] = (
    weights["fidelity"] * explanations["fidelity"]
    + weights["usefulness"] * explanations["usefulness"]
    + weights["actionability"] * explanations["actionability"]
    + weights["stability"] * explanations["stability"]
)

explanations["user_requested_review"] = rng.binomial(
    1,
    p=np.clip(0.45 - 0.35 * explanations["explanation_quality"], 0.05, 0.80),
    size=n,
)

explanations.groupby("risk_level").agg(
    mean_explanation_quality=("explanation_quality", "mean"),
    review_request_rate=("user_requested_review", "mean"),
).reset_index()

## Interpretation

Better explanations should reduce confusion, but high-risk systems still need review pathways even when explanations are clear.